In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


In [2]:
dataset = pd.read_csv("CKD.csv")

print(dataset.shape)
print(dataset.head())
print(dataset.info())

(399, 25)
   age         bp sg   al   su     rbc        pc         pcc          ba  \
0  2.0  76.459948  c  3.0  0.0  normal  abnormal  notpresent  notpresent   
1  3.0  76.459948  c  2.0  0.0  normal    normal  notpresent  notpresent   
2  4.0  76.459948  a  1.0  0.0  normal    normal  notpresent  notpresent   
3  5.0  76.459948  d  1.0  0.0  normal    normal  notpresent  notpresent   
4  5.0  50.000000  c  0.0  0.0  normal    normal  notpresent  notpresent   

          bgr  ...        pcv            wc        rc  htn  dm  cad  appet  \
0  148.112676  ...  38.868902   8408.191126  4.705597   no  no   no    yes   
1  148.112676  ...  34.000000  12300.000000  4.705597   no  no   no    yes   
2   99.000000  ...  34.000000   8408.191126  4.705597   no  no   no    yes   
3  148.112676  ...  38.868902   8408.191126  4.705597   no  no   no    yes   
4  148.112676  ...  36.000000  12400.000000  4.705597   no  no   no    yes   

     pe  ane classification  
0   yes   no            yes  
1  p

In [3]:
print("Missing Values:")
print(dataset.isnull().sum())

print("Duplicates:", dataset.duplicated().sum())


Missing Values:
age               0
bp                0
sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sc                0
sod               0
pot               0
hrmo              0
pcv               0
wc                0
rc                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64
Duplicates: 0


In [4]:
X = dataset.drop("classification", axis=1)
y = (dataset["classification"] == "yes").astype(int)
# Splitdata
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:

categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    exclude=["object"]
).columns.tolist()


In [6]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ), categorical_cols)
])


In [7]:

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),
    "SVM": SVC(kernel="rbf")
}

In [8]:
results = []
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

    trained_models[name] = pipeline

In [9]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

          Algorithm  Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.9875        1.0    0.98  0.989899
      Decision Tree    0.9875        1.0    0.98  0.989899
      Random Forest    0.9875        1.0    0.98  0.989899
  Gradient Boosting    0.9875        1.0    0.98  0.989899
                SVM    0.9875        1.0    0.98  0.989899


In [10]:
final_model = trained_models["Logistic Regression"]

final_prediction = final_model.predict(X_test)

print("\nFinal Model: Logistic Regression")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_prediction))

print("\nClassification Report:")
print(classification_report(
    y_test,
    final_prediction,
    target_names=["No CKD", "CKD"]
))



Final Model: Logistic Regression

Confusion Matrix:
[[30  0]
 [ 1 49]]

Classification Report:
              precision    recall  f1-score   support

      No CKD       0.97      1.00      0.98        30
         CKD       1.00      0.98      0.99        50

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80

